In [ ]:
%matplotlib inline

import dask.array as da
import numpy as np
from numba import jit
from pyscenarios import sobol, clusterization, plot_couples
from pyscenarios.scramble import scramble

# Shared configuration
samples = 8191
chunksize=1024
dimensions = 6
d0 = 10

## Mersenne Twister

Generate one plot for each unique combination of the cartesian product of all dimensions

In [ ]:
rng = da.random.RandomState(d0)
M = rng.random((samples, dimensions), chunks=chunksize)
plot_couples(M, d0=d0, heat=clusterization(M))

## Vanilla Joe/Kuo

In [ ]:
S = sobol((samples, dimensions), d0=d0, chunks=chunksize)
plot_couples(S, d0=d0, heat=clusterization(S))

## mod(S + M, 1)

Where
- S is the vanilla Joe/Kuo series with shape=(series, dimensions)
- M is a 1-dimensional Mersenne Twister series with shape=(dimensions, )

In [ ]:
rng = np.random.RandomState(0)
shift = rng.random(dimensions)
S2 = (S + shift) % 1.0
plot_couples(S2, d0=d0, heat=clusterization(S2))

## Binary XOR

In [ ]:
# WARNING: If you tamper with this, make sure to restart Jupyter Notebook.
# The @jit decorator below will capture the contents of shift_int at the
# moment of the first compilation and not notice later changes.
rng = np.random.RandomState(0)
shift_int = rng.randint(int(2 ** 32), size=dimensions, dtype="u4")


# Copy-paste from pyscenarios.sobol
@jit(
    "void(uint32, uint32, uint32, uint32, uint32[:, :], float64[:, :])",
    nopython=True,
    nogil=True,
    cache=False,
)
def _sobol_kernel_jit(
    samples: int, dimensions: int, s0: int, d0: int, V: np.ndarray, output: np.ndarray
) -> None:

    for j in range(dimensions):
        state = 0
        for i in range(s0 + samples):
            # c = index from the right of the first zero bit of i
            c = 0
            mask = 1
            while i & mask:
                mask *= 2
                c += 1

            state ^= V[j + d0, c]

            # Start patch
            shifted_state = state ^ shift_int[j]
            if i >= s0:
                output[i - s0, j] = np.double(shifted_state) / np.double(2 ** 32)
            # End patch


# Monkey-patch, regenerate series, and undo patch
import sys

mod = sys.modules["pyscenarios.sobol"]
bak = mod._sobol_kernel_jit
try:
    mod._sobol_kernel_jit = _sobol_kernel_jit
    S3 = sobol((samples, dimensions), d0=d0)
finally:
    mod._sobol_kernel_jit = bak
S3 = da.from_array(S3, chunks=chunksize)

In [ ]:
plot_couples(S3, d0=d0, heat=clusterization(S3))

## Display traslation caused by XOR

In [ ]:
plot_couples(S3 - S, d0=d0, limits=[-1, 1])

# Owen scramble

In [ ]:
S4 = S.rechunk(-1).map_blocks(scramble).rechunk(chunksize)
plot_couples(S4, d0=d0, heat=clusterization(S4))